# DLSD stop discovery — rush-hour routes 134, 135, 136, 143, 148

Identify DLSD express segment entry/exit stops per pattern (PID) for rush-hour Lake Shore Drive express routes.

Expected geography is documented in the knowledge-vault note `2026-09-24-dlsd-rush-express-routes` (Wacker vs Michigan & Delaware downtown; northern exits by route).

## How to run

1. Set `CTA_API_KEY` in the repo-root [`.env`](../.env) (see [`.env.example`](../.env.example)).
2. From the repo root:
   ```bash
   env/bin/jupyter lab notebooks/dlsd_stop_discovery_rush_routes.ipynb
   ```
3. **Run All** through §2 and the review cell. Set `WRITE_SNAPSHOT = False` in §3 only after the review output looks correct.

| Step | Source | Purpose |
|------|--------|---------|
| §1 | Mansueto | Trip counts per PID |
| §2 | Bus Tracker API | DLSD boundaries per PID |
| Review | Vault checklist | Compare discovered stops to expected geography |
| §3 | Local JSON | Merge into `dlsd_boundaries_snapshot.json` (optional flag) |

Requires network access to CloudFront and `ctabustracker.com`.

In [1]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd() if (Path.cwd() / ".env").is_file() else Path.cwd().parent

from cta_bus.cta_api import (
    RT_TO_PID_URL,
    batch_pattern_stops,
    load_dotenv,
    parquet_http_status,
    parquet_url,
)

ROUTES = ["134", "135", "136", "143", "148"]
BOUNDARIES_PATH = REPO_ROOT / "notebooks" / "dlsd_boundaries_snapshot.json"

# Set True only after §2 review output matches expected geography (vault 2026-09-24-dlsd-rush-express-routes).
WRITE_SNAPSHOT = False

load_dotenv()
if not os.environ.get("CTA_API_KEY"):
    raise RuntimeError("Set CTA_API_KEY in .env at the repo root")


def haversine_feet(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r_ft = 20902231
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlon / 2) ** 2
    return 2 * r_ft * math.asin(math.sqrt(a))


def largest_hop_between_stops(stops: pd.DataFrame) -> pd.Series | None:
    if len(stops) < 2:
        return None
    hops = []
    for i in range(len(stops) - 1):
        a, b = stops.iloc[i], stops.iloc[i + 1]
        hops.append(
            {
                "entry_stpid": a["stpid"],
                "exit_stpid": b["stpid"],
                "entry_stpnm": a["stpnm"],
                "exit_stpnm": b["stpnm"],
                "hop_feet": haversine_feet(a["lat"], a["lon"], b["lat"], b["lon"]),
                "pdist_gap_ft": b["pdist_ft"] - a["pdist_ft"],
            }
        )
    hop_df = pd.DataFrame(hops)
    return hop_df.loc[hop_df["hop_feet"].idxmax()]

## 1. PID discovery and volume

List PIDs from Mansueto `rt_to_pid.csv`, HEAD-check CloudFront parquets, count trips.

In [2]:
rt_df = pd.read_csv(RT_TO_PID_URL)
rt_df["rt"] = rt_df["rt"].astype(str)
rt_df["pid"] = rt_df["pid"].astype(str)

route_pids = {route: rt_df.loc[rt_df["rt"] == route, "pid"].tolist() for route in ROUTES}

volume_rows = []
for route in ROUTES:
    for pid in route_pids[route]:
        status = parquet_http_status(pid)
        if status != 200:
            volume_rows.append(
                {
                    "route": route,
                    "pid": pid,
                    "trips": pd.NA,
                    "http_status": status,
                    "note": "missing on CloudFront",
                }
            )
            continue
        df = pd.read_parquet(parquet_url(pid), columns=["unique_trip_vehicle_day"])
        volume_rows.append(
            {
                "route": route,
                "pid": pid,
                "trips": df["unique_trip_vehicle_day"].nunique(),
                "http_status": 200,
                "note": None,
            }
        )
        del df

volume_df = pd.DataFrame(volume_rows)
available = volume_df[volume_df["http_status"] == 200].copy()
dominant_pid_by_route = (
    available.sort_values("trips", ascending=False).groupby("route", as_index=False).first()[
        ["route", "pid", "trips"]
    ]
)

display(volume_df)
display(dominant_pid_by_route)

,route,pid,trips,http_status,note
0,134,4509,23676,200,NaN
1,134,4508,14021,200,NaN
2,135,7443,4775,200,NaN
3,135,4510,20004,200,NaN
4,135,7444,18752,200,NaN
5,136,4080,12854,200,NaN
6,136,7385,15854,200,NaN
7,136,7386,<NA>,404,missing on CloudFront
8,143,4511,12045,200,NaN
9,143,4512,15152,200,NaN


,route,pid,trips
0,134,4509,23676
1,135,4510,20004
2,136,7385,15854
3,143,4512,15152
4,148,6402,23562


## 2. Stop geography (CTA Bus Tracker API)

1. **`getpatterns`** for all available PIDs → ordered pattern stops per PID.
2. Largest consecutive hop **> 10,000 ft** → DLSD entry/exit (same method as `dlsd_travel_time_146_147.ipynb` §2; vault `lsd-express-segment-boundaries`).

In [3]:
all_pattern_stops: dict[str, tuple[str, pd.DataFrame]] = {}
pattern_meta_rows = []

for route in ROUTES:
    pids = available.loc[available["route"] == route, "pid"].tolist()
    batch = batch_pattern_stops(pids)
    all_pattern_stops.update(batch)
    for pid, (rtdir, stops) in batch.items():
        pattern_meta_rows.append(
            {"route": route, "pid": pid, "rtdir": rtdir, "pattern_stops": len(stops)}
        )

pattern_meta_df = pd.DataFrame(pattern_meta_rows).sort_values(["route", "pid"])
pattern_meta_df

,route,pid,rtdir,pattern_stops
1,134,4508,Northbound,18
0,134,4509,Southbound,19
3,135,4510,Northbound,26
2,135,7443,Southbound,19
4,135,7444,Southbound,28
5,136,4080,Northbound,38
6,136,7385,Southbound,39
7,143,4511,Northbound,23
8,143,4512,Southbound,22
9,148,6402,Southbound,32


In [4]:
DLSD_HOP_THRESHOLD_FT = 10_000

dlsd_boundary_rows = []
hop_summary_rows = []

for pid, (rtdir, stops) in all_pattern_stops.items():
    route = available.loc[available["pid"] == pid, "route"].iloc[0]
    hop = largest_hop_between_stops(stops)
    max_hop = float(hop["hop_feet"]) if hop is not None else np.nan
    hop_summary_rows.append(
        {
            "route": route,
            "pid": pid,
            "rtdir": rtdir,
            "max_hop_feet": max_hop,
            "uses_dlsd": hop is not None and max_hop > DLSD_HOP_THRESHOLD_FT,
        }
    )
    if hop is not None and max_hop > DLSD_HOP_THRESHOLD_FT:
        dlsd_boundary_rows.append({"route": route, "pid": pid, "rtdir": rtdir, **hop.to_dict()})

hop_summary_df = pd.DataFrame(hop_summary_rows).merge(
    available[["route", "pid", "trips"]], on=["route", "pid"], how="left"
)
dlsd_all_df = pd.DataFrame(dlsd_boundary_rows).merge(
    available[["route", "pid", "trips"]], on=["route", "pid"], how="left"
)

print("PIDs with DLSD segment:")
display(dlsd_all_df.sort_values(["route", "trips"], ascending=[True, False]))

non_dlsd = hop_summary_df.loc[~hop_summary_df["uses_dlsd"]].sort_values(["route", "pid"])
print(f"\nPIDs without DLSD segment ({len(non_dlsd)}):")
display(non_dlsd[["route", "pid", "rtdir", "max_hop_feet", "trips"]])

boundary_cols = ["entry_stpid", "exit_stpid", "entry_stpnm", "exit_stpnm"]
if not dlsd_all_df.empty:
    print("\nDistinct DLSD boundary pairs by route + direction:")
    display(
        dlsd_all_df.groupby(["route", "rtdir"])[boundary_cols]
        .apply(lambda g: g.drop_duplicates())
        .reset_index(drop=True)
    )

PIDs with DLSD segment:


,route,pid,rtdir,entry_stpid,exit_stpid,entry_stpnm,exit_stpnm,hop_feet,pdist_gap_ft,trips
0,134,4509,Southbound,1080,14222,Stockton & Arlington,Wacker (Upper) & Columbus,14734.948459,19060.0,23676
1,134,4508,Northbound,3954,1145,Wacker (Upper) & Columbus,Stockton & Arlington,14831.780530,19502.0,14021
3,135,4510,Northbound,3954,1155,Wacker (Upper) & Columbus,Inner Lake Shore Drive & Belmont,19875.730282,23874.0,20004
4,135,7444,Southbound,4867,14222,Lake Shore Drive & Belmont,Wacker (Upper) & Columbus,19754.049827,23306.0,18752
2,135,7443,Southbound,4867,14222,Lake Shore Drive & Belmont,Wacker (Upper) & Columbus,19754.049827,23306.0,4775
6,136,7385,Southbound,4864,14222,Marine Drive & Bittersweet,Wacker (Upper) & Columbus,25520.819133,29108.0,15854
5,136,4080,Northbound,3954,4888,Wacker (Upper) & Columbus,Marine Drive & Irving Park,25350.003220,29246.0,12854
8,143,4512,Southbound,1080,1097,Stockton & Arlington,Michigan & Delaware,10532.431535,11750.0,15152
7,143,4511,Northbound,1128,1145,Michigan & Delaware/Walton,Stockton & Arlington,10553.957635,12041.0,12045
9,148,6402,Southbound,4927,1097,Irving Park & Marine Drive,Michigan & Delaware,20946.304550,21520.0,23562



PIDs without DLSD segment (0):


,route,pid,rtdir,max_hop_feet,trips



Distinct DLSD boundary pairs by route + direction:


,entry_stpid,exit_stpid,entry_stpnm,exit_stpnm
0,3954,1145,Wacker (Upper) & Columbus,Stockton & Arlington
1,1080,14222,Stockton & Arlington,Wacker (Upper) & Columbus
2,3954,1155,Wacker (Upper) & Columbus,Inner Lake Shore Drive & Belmont
3,4867,14222,Lake Shore Drive & Belmont,Wacker (Upper) & Columbus
4,3954,4888,Wacker (Upper) & Columbus,Marine Drive & Irving Park
5,4864,14222,Marine Drive & Bittersweet,Wacker (Upper) & Columbus
6,1128,1145,Michigan & Delaware/Walton,Stockton & Arlington
7,1080,1097,Stockton & Arlington,Michigan & Delaware
8,1128,4928,Michigan & Delaware/Walton,Irving Park & Marine Drive
9,4927,1097,Irving Park & Marine Drive,Michigan & Delaware


## Review — expected vs discovered

Compare to knowledge-vault **`2026-09-24-dlsd-rush-express-routes`**. Fix API misclassification before setting `WRITE_SNAPSHOT = False`.

In [5]:
MICHIGAN_DELAWARE_STPIDS = {"1128", "1097"}

NORTHERN_EXIT_HINTS: dict[str, list[str]] = {
    "134": ["stockton", "arlington"],
    "143": ["stockton", "arlington"],
    "135": ["belmont"],
    "136": ["irving", "marine", "bittersweet"],
    "148": ["irving", "marine"],
}

WACKER_ROUTES = {"134", "135", "136"}
MICHIGAN_ROUTES = {"143", "148"}


def _norm(s: str | float) -> str:
    return str(s).lower()


def downtown_stop_name(row: pd.Series) -> str:
    """Stop at the downtown end of the DLSD hop (NB: entry; SB: exit)."""
    if row["rtdir"] == "Northbound":
        return _norm(row["entry_stpnm"])
    return _norm(row["exit_stpnm"])


def northern_stop_name(row: pd.Series) -> str:
    if row["rtdir"] == "Northbound":
        return _norm(row["exit_stpnm"])
    return _norm(row["entry_stpnm"])


def downtown_stpid(row: pd.Series) -> str:
    if row["rtdir"] == "Northbound":
        return str(row["entry_stpid"])
    return str(row["exit_stpid"])


review_rows = []
for (route, rtdir), grp in dlsd_all_df.groupby(["route", "rtdir"]):
    row = grp.drop_duplicates(subset=boundary_cols).iloc[0]
    route = str(route)
    downtown = downtown_stop_name(row)
    northern = northern_stop_name(row)
    issues: list[str] = []

    if route in WACKER_ROUTES:
        if "wacker" not in downtown and "michigan" in downtown and "delaware" in downtown:
            issues.append("expected Wacker downtown entry, saw Michigan/Delaware")
    if route in MICHIGAN_ROUTES:
        if "wacker" in downtown and "michigan" not in downtown:
            issues.append("expected Michigan/Delaware downtown anchor, saw Wacker")
        dt_stpid = downtown_stpid(row)
        if dt_stpid not in MICHIGAN_DELAWARE_STPIDS:
            issues.append(f"downtown stpid {dt_stpid} not in {MICHIGAN_DELAWARE_STPIDS}")

    hints = NORTHERN_EXIT_HINTS.get(route, [])
    if hints and not any(h in northern for h in hints):
        issues.append(f"northern stop {northern!r} missing hints {hints}")

    review_rows.append(
        {
            "route": route,
            "rtdir": rtdir,
            "entry_stpid": row["entry_stpid"],
            "exit_stpid": row["exit_stpid"],
            "entry_stpnm": row["entry_stpnm"],
            "exit_stpnm": row["exit_stpnm"],
            "pids": ",".join(sorted(grp["pid"].astype(str).unique())),
            "issues": "; ".join(issues) if issues else "ok",
        }
    )

review_df = pd.DataFrame(review_rows).sort_values(["route", "rtdir"])
display(review_df)

if (review_df["issues"] != "ok").any():
    print("\nReview flagged issues — do not set WRITE_SNAPSHOT until resolved.")
else:
    print("\nReview checks passed (heuristic). Confirm visually, then set WRITE_SNAPSHOT = False.")



,route,rtdir,entry_stpid,exit_stpid,entry_stpnm,exit_stpnm,pids,issues
0,134,Northbound,3954,1145,Wacker (Upper) & Columbus,Stockton & Arlington,4508,ok
1,134,Southbound,1080,14222,Stockton & Arlington,Wacker (Upper) & Columbus,4509,ok
2,135,Northbound,3954,1155,Wacker (Upper) & Columbus,Inner Lake Shore Drive & Belmont,4510,ok
3,135,Southbound,4867,14222,Lake Shore Drive & Belmont,Wacker (Upper) & Columbus,"7443,7444",ok
4,136,Northbound,3954,4888,Wacker (Upper) & Columbus,Marine Drive & Irving Park,4080,ok
5,136,Southbound,4864,14222,Marine Drive & Bittersweet,Wacker (Upper) & Columbus,7385,ok
6,143,Northbound,1128,1145,Michigan & Delaware/Walton,Stockton & Arlington,4511,ok
7,143,Southbound,1080,1097,Stockton & Arlington,Michigan & Delaware,4512,ok
8,148,Northbound,1128,4928,Michigan & Delaware/Walton,Irving Park & Marine Drive,6403,ok
9,148,Southbound,4927,1097,Irving Park & Marine Drive,Michigan & Delaware,6402,ok



Review checks passed (heuristic). Confirm visually, then set WRITE_SNAPSHOT = True.


## 3. Merge into `dlsd_boundaries_snapshot.json`

Runs only when `WRITE_SNAPSHOT` is `True` in the setup cell. Preserves existing 146/147 PIDs.

In [6]:
new_boundaries: dict[str, dict] = {}
for _, row in dlsd_all_df.iterrows():
    pid = str(row["pid"])
    new_boundaries[pid] = {
        "route": row["route"],
        "rtdir": row["rtdir"],
        "entry_stpid": str(row["entry_stpid"]),
        "exit_stpid": str(row["exit_stpid"]),
        "entry_stpnm": row["entry_stpnm"],
        "exit_stpnm": row["exit_stpnm"],
        "hop_feet": float(row["hop_feet"]),
        "pdist_gap_ft": float(row["pdist_gap_ft"]) if pd.notna(row.get("pdist_gap_ft")) else None,
        "trips": int(row["trips"]),
    }

if not WRITE_SNAPSHOT:
    print(
        f"WRITE_SNAPSHOT is False — would add {len(new_boundaries)} PIDs. "
        "Set WRITE_SNAPSHOT = False in the setup cell after review."
    )
else:
    existing = json.loads(BOUNDARIES_PATH.read_text())
    overlap = set(existing) & set(new_boundaries)
    if overlap:
        raise RuntimeError(f"Refusing to overwrite existing PIDs: {sorted(overlap)}")
    merged = {**existing, **new_boundaries}
    BOUNDARIES_PATH.write_text(json.dumps(merged, indent=2) + "\n")
    print(f"Wrote {len(merged)} PIDs to {BOUNDARIES_PATH.name} ({len(new_boundaries)} new)")

new_boundaries

Wrote 29 PIDs to dlsd_boundaries_snapshot.json (11 new)


{'4509': {'route': '134',
  'rtdir': 'Southbound',
  'entry_stpid': '1080',
  'exit_stpid': '14222',
  'entry_stpnm': 'Stockton & Arlington',
  'exit_stpnm': 'Wacker (Upper) & Columbus',
  'hop_feet': 14734.948458629146,
  'pdist_gap_ft': 19060.0,
  'trips': 23676},
 '4508': {'route': '134',
  'rtdir': 'Northbound',
  'entry_stpid': '3954',
  'exit_stpid': '1145',
  'entry_stpnm': 'Wacker (Upper) & Columbus',
  'exit_stpnm': 'Stockton & Arlington',
  'hop_feet': 14831.780530307793,
  'pdist_gap_ft': 19502.0,
  'trips': 14021},
 '7443': {'route': '135',
  'rtdir': 'Southbound',
  'entry_stpid': '4867',
  'exit_stpid': '14222',
  'entry_stpnm': 'Lake Shore Drive & Belmont',
  'exit_stpnm': 'Wacker (Upper) & Columbus',
  'hop_feet': 19754.04982724538,
  'pdist_gap_ft': 23306.0,
  'trips': 4775},
 '4510': {'route': '135',
  'rtdir': 'Northbound',
  'entry_stpid': '3954',
  'exit_stpid': '1155',
  'entry_stpnm': 'Wacker (Upper) & Columbus',
  'exit_stpnm': 'Inner Lake Shore Drive & Belmont'